<a href="https://colab.research.google.com/github/goitstudent123/llm-gen/blob/main/dz_topic_7_has.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Chunk 1: Install dependencies
!pip -q install -U langchain langchain-openai langchain-chroma langchain-text-splitters chromadb
!pip -q check

In [2]:
# Chunk 2: Initial imports and OpenRouter configuration
import os
from typing import List

from IPython.display import Markdown, display

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


def read_colab_secret(name: str) -> str | None:
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

openrouter_api_key = (
    read_colab_secret("OPENROUTER_API_KEY")
    or os.getenv("OPENROUTER_API_KEY")
)

if not openrouter_api_key:
    raise ValueError(
        "Add OPENROUTER_API_KEY to Colab Secrets or set it in os.environ."
    )

OPENROUTER_CHAT_MODEL = (
    read_colab_secret("OPENROUTER_CHAT_MODEL")
    or os.getenv("OPENROUTER_CHAT_MODEL")
    or "openai/gpt-4.1-mini"
)

OPENROUTER_EMBEDDING_MODEL = (
    read_colab_secret("OPENROUTER_EMBEDDING_MODEL")
    or os.getenv("OPENROUTER_EMBEDDING_MODEL")
    or "openai/text-embedding-3-small"
)

OPENROUTER_HTTP_REFERER = (
    read_colab_secret("OPENROUTER_HTTP_REFERER")
    or os.getenv("OPENROUTER_HTTP_REFERER")
    or "https://colab.research.google.com"
)

OPENROUTER_APP_TITLE = (
    read_colab_secret("OPENROUTER_APP_TITLE")
    or os.getenv("OPENROUTER_APP_TITLE")
    or "Colab Financial RAG"
)

In [3]:
# Chunk 3: Source financial reports text
financial_reports_text = """
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.

ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.

РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.

ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує у масштабування бізнесу.
""".strip()

In [6]:
# Chunk 4: Split text into logical company chunks
text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n"],
    chunk_size=350,
    chunk_overlap=0,
    keep_separator=False,
)

documents = text_splitter.create_documents([financial_reports_text])

print(f"Total chunks: {len(documents)}")

for index, document in enumerate(documents, 1):
    print("=" * 80)
    print(f"Chunk {index}")
    print(document.page_content)

Total chunks: 4
Chunk 1
ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.
Chunk 2
ГрінЕнерджі Солюшнс за 2024 рік: виручка 890 млн дол (+35 відсотків), чистий прибуток 156 млн дол, маржа EBITDA 28 відсотків.
Працівників 5100 осіб. ROE становить 19.7 відсотка. Співвідношення борг до EBITDA дорівнює 0.9. Лідер у секторі відновлюваної енергії.
Chunk 3
РітейлМакс Груп за 2024 рік: виручка 1200 млн дол (+8 відсотків), чистий прибуток 48 млн дол, маржа EBITDA 12 відсотків.
Працівників 8500 осіб. ROE становить 9.3 відсотка. Співвідношення борг до EBITDA дорівнює 3.2. Мережа з 340 магазинів в Україні.
Chunk 4
ФінТех Дайнамікс за 2024 рік: виручка 230 млн дол (+52 відсотки), чистий збиток мінус 15 млн дол, маржа EBITDA мінус 8 відсотків.
Працівників 890 осіб. Від'ємний показник ROE. Стартап який активно інвестує

In [7]:
# Chunk 5: Validate chunking acceptance criteria
expected_company_names = [
    "ТехНова Інк",
    "ГрінЕнерджі Солюшнс",
    "РітейлМакс Груп",
    "ФінТех Дайнамікс",
]

assert len(documents) == 4, f"Expected 4 chunks, got {len(documents)}"

for company_name, document in zip(expected_company_names, documents):
    assert company_name in document.page_content, f"Missing company in chunk: {company_name}"

print("Chunking validation passed.")

Chunking validation passed.


In [8]:
# Chunk 6: Create OpenRouter chat model and embeddings
common_openrouter_headers = {
    "HTTP-Referer": OPENROUTER_HTTP_REFERER,
    "X-OpenRouter-Title": OPENROUTER_APP_TITLE,
}

chat_model = ChatOpenAI(
    model=OPENROUTER_CHAT_MODEL,
    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    default_headers=common_openrouter_headers,
)

embeddings = OpenAIEmbeddings(
    model=OPENROUTER_EMBEDDING_MODEL,
    api_key=openrouter_api_key,
    base_url=OPENROUTER_BASE_URL,
    default_headers=common_openrouter_headers,
)

print(f"Chat model: {OPENROUTER_CHAT_MODEL}")
print(f"Embedding model: {OPENROUTER_EMBEDDING_MODEL}")

Chat model: openai/gpt-4.1-mini
Embedding model: openai/text-embedding-3-small


In [10]:
# Chunk 7: Create Chroma vector store
COLLECTION_NAME = "finansovi_zvity"
COLLECTION_DISPLAY_NAME_UK = "фінансові_звіти"

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    collection_metadata={
        "display_name_uk": COLLECTION_DISPLAY_NAME_UK,
        "description": "Financial reports of Ukrainian companies",
    },
)

print(f"Chroma collection technical name: {COLLECTION_NAME}")
print(f"Chroma collection Ukrainian display name: {COLLECTION_DISPLAY_NAME_UK}")

Chroma collection technical name: finansovi_zvity
Chroma collection Ukrainian display name: фінансові_звіти


In [11]:
# Chunk 8: Configure retriever to return top 2 relevant chunks
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 2,
    }
)

print("Retriever is configured with k=2.")

Retriever is configured with k=2.


In [12]:
# Chunk 9: Build RAG chain
def format_documents(retrieved_documents: List[Document]) -> str:
    return "\n\n".join(document.page_content for document in retrieved_documents)


prompt = ChatPromptTemplate.from_template(
    """
Ти фінансовий RAG-асистент.

Відповідай українською мовою.
Використовуй тільки наданий контекст.
Якщо в контексті немає точної інформації про компанію або показник, чесно скажи, що інформація відсутня.
Не вигадуй факти.
Відповідай коротко, але з конкретними цифрами, якщо вони є.

Контекст:
{context}

Питання:
{question}

Відповідь:
""".strip()
)

rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough(),
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

print("RAG chain is ready.")

RAG chain is ready.


In [13]:
# Chunk 10: Test retrieval quality manually
debug_question = "Яка виручка у компанії ТехНова Інк за 2024 рік?"

retrieved_documents = retriever.invoke(debug_question)

display(Markdown(f"### Debug question\n{debug_question}"))

for index, document in enumerate(retrieved_documents, 1):
    display(Markdown(f"#### Retrieved chunk {index}"))
    print(document.page_content)

### Debug question
Яка виручка у компанії ТехНова Інк за 2024 рік?

#### Retrieved chunk 1

ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.


#### Retrieved chunk 2

ТехНова Інк за 2024 рік: виручка 450 млн дол (+18 відсотків), чистий прибуток 67 млн дол, маржа EBITDA 22 відсотки.
Працівників 2300 осіб. ROE становить 15.2 відсотка. Співвідношення борг до EBITDA дорівнює 1.8. Штаб-квартира у Львові.


In [14]:
# Chunk 11: Run acceptance tests
questions = [
    "Яка виручка у компанії ТехНова Інк за 2024 рік?",
    "Скільки працівників у ГрінЕнерджі Солюшнс?",
    "Який показник ROE у РітейлМакс Груп?",
    "Де знаходиться штаб-квартира ТехНова Інк?",
    "Яка маржа EBITDA у ФінТех Дайнамікс?",
    "Який чистий прибуток у ЕкоПакування Солюшнс?",
]

for index, question in enumerate(questions, 1):
    print("=" * 70)
    print(f"Питання {index}: {question}")

    answer = rag_chain.invoke(question)

    print(f"Відповідь: {answer}")
    print("-" * 70)

Питання 1: Яка виручка у компанії ТехНова Інк за 2024 рік?
Відповідь: Виручка компанії ТехНова Інк за 2024 рік становить 450 млн дол.
----------------------------------------------------------------------
Питання 2: Скільки працівників у ГрінЕнерджі Солюшнс?
Відповідь: У ГрінЕнерджі Солюшнс працює 5100 осіб.
----------------------------------------------------------------------
Питання 3: Який показник ROE у РітейлМакс Груп?
Відповідь: ROE у РітейлМакс Груп становить 9.3 відсотка.
----------------------------------------------------------------------
Питання 4: Де знаходиться штаб-квартира ТехНова Інк?
Відповідь: Штаб-квартира ТехНова Інк знаходиться у Львові.
----------------------------------------------------------------------
Питання 5: Яка маржа EBITDA у ФінТех Дайнамікс?
Відповідь: Маржа EBITDA у ФінТех Дайнамікс за 2024 рік становить мінус 8 відсотків.
----------------------------------------------------------------------
Питання 6: Який чистий прибуток у ЕкоПакування Солюшнс?
В